# 05 · Per-image metadata

**Job:** write one metadata record for each of the 10,200 images.

| Split | Objects | Colours, shapes, context | Relations |
|---|---|---|---|
| KB | COCO human labels | keywords in its 5 KB captions | box-overlap pairs |
| Validation, test | DETR detector (`facebook/detr-resnet-50`, score ≥ 0.7) | not filled (test captions are never read) | box-overlap pairs |

Every field says where it comes from, so nobody mistakes a heuristic for ground truth.

| Output | Where |
|---|---|
| Full records | `data/metadata/image_metadata.json` |
| Flat table | `data/metadata/image_metadata.csv` |
| Summary | `results/qa/metadata_summary.json` |

Run on Google Colab. All files are read from and written to `MyDrive/G9_Hybrid_RAG/`. Needs notebooks 01–04. **Use a T4 GPU.**

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
%pip -q install faiss-cpu rank-bm25 timm

Mounted at /content/drive


In [ ]:
import json, re, time, zipfile
from io import BytesIO
from pathlib import Path
from itertools import combinations
from collections import defaultdict
import numpy as np, pandas as pd, torch, faiss, transformers
from PIL import Image, ImageOps
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from transformers import CLIPModel, CLIPImageProcessor, AutoTokenizer

SEED = 42
rng = np.random.default_rng(SEED)
ROOT = Path('/content/drive/MyDrive/G9_Hybrid_RAG')
DATA = ROOT/'data'
EMB = DATA/'embeddings/clip_vit_base_patch32'
IDX = DATA/'indexes'
ARCH = DATA/'images_archives'
QA = ROOT/'results/qa'
RELEASE = ROOT/'results/preprocessing_release'
META = DATA/'metadata'
for d in (QA, RELEASE, META): d.mkdir(parents=True, exist_ok=True)

with open(DATA/'splits/split_ids.json') as f: splits = json.load(f)
SPLITS = ['knowledge_base', 'validation', 'test']
ids = {s: sorted(map(int, splits[s])) for s in SPLITS}
manifest = pd.read_csv(DATA/'splits/image_manifest.csv')
fname = {int(r.image_id): f"{r.split}/{r.file_name}" for r in manifest.itertuples()}
split_of = {int(r.image_id): r.split for r in manifest.itertuples()}

caps = pd.read_csv(EMB/'kb_text_metadata.csv')          # row i <-> text embedding i <-> text index position i
img_ids = np.load(EMB/'kb_image_ids.npy')               # row i <-> image embedding i <-> image index position i
img_emb = np.load(EMB/'kb_image_embeddings.npy')
txt_emb = np.load(EMB/'kb_text_embeddings.npy')
assert img_emb.shape == (10000, 512) and txt_emb.shape == (50000, 512) and len(caps) == 50000
assert (np.diff(img_ids) > 0).all(), 'image ids must be sorted ascending'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
CLIP_NAME = 'openai/clip-vit-base-patch32'
clip = CLIPModel.from_pretrained(CLIP_NAME).to(device).eval()
proc = CLIPImageProcessor.from_pretrained(CLIP_NAME)
tok = AutoTokenizer.from_pretrained(CLIP_NAME)

def feat(o):
    # transformers 4.x returns a tensor, 5.x returns an output object; handle both
    return o.pooler_output if hasattr(o, 'pooler_output') else o

@torch.inference_mode()
def embed_images(images, bs=32):
    out = []
    for i in range(0, len(images), bs):
        px = proc(images=images[i:i+bs], return_tensors='pt')['pixel_values'].to(device)
        f = feat(clip.get_image_features(pixel_values=px)).float()
        out.append(torch.nn.functional.normalize(f, dim=-1).cpu().numpy())
    return np.concatenate(out).astype('float32')

@torch.inference_mode()
def embed_texts(texts, bs=128):
    out = []
    for i in range(0, len(texts), bs):
        t = tok(texts[i:i+bs], padding=True, truncation=True, max_length=77, return_tensors='pt').to(device)
        f = feat(clip.get_text_features(**t)).float()
        out.append(torch.nn.functional.normalize(f, dim=-1).cpu().numpy())
    return np.concatenate(out).astype('float32')

def zip_index(split):
    d = {}
    for zp in sorted((ARCH/split).glob('part_*.zip')):
        with zipfile.ZipFile(zp) as z:
            for n in z.namelist():
                if not n.endswith('/'): d[n] = zp
    return d
ZIPS = {s: zip_index(s) for s in SPLITS}

def load_image(image_id):
    name = fname[image_id]
    with zipfile.ZipFile(ZIPS[name.split('/')[0]][name]) as z, Image.open(BytesIO(z.read(name))) as im:
        return ImageOps.exif_transpose(im).convert('RGB')

print('device:', device, '| transformers', transformers.__version__)
print({s: len(ids[s]) for s in SPLITS})

device: cuda | transformers 5.22.1
{'knowledge_base': 10000, 'validation': 100, 'test': 100}


## 2. KB images: COCO labels + caption keywords

In [ ]:
COLORS = ['black','white','red','blue','green','yellow','brown','orange','pink','purple','gray','grey']
SHAPES = ['round','circular','square','rectangular','oval','triangular','long','tall','flat']
CTX = {
    'indoor':  {'indoor','indoors','kitchen','bathroom','bedroom','living','room','office'},
    'outdoor': {'outdoor','outdoors','street','beach','mountain','park','field','road','sidewalk'},
    'night':   {'night','nighttime','evening','dark'},
    'day':     {'daytime','daylight','sunny','morning','afternoon'},
}

def caption_cues(captions):
    toks = set(w for c in captions for w in re.findall(r"[a-z]+", c.lower()))
    return {
        'colors': sorted(c for c in COLORS if c in toks),
        'shapes': sorted(s for s in SHAPES if s in toks),
        'context': {k: bool(v & toks) for k, v in CTX.items()},
    }

def overlap_pairs(objs, max_pairs=10):
    # objs: list of (label, x, y, w, h). Geometric proxy only - NOT a semantic relation.
    seen, n = {}, 0
    for (l1,x1,y1,w1,h1),(l2,x2,y2,w2,h2) in combinations(objs, 2):
        if min(x1+w1, x2+w2) > max(x1, x2) and min(y1+h1, y2+h2) > max(y1, y2):
            n += 1
            key = tuple(sorted((l1, l2)))
            seen[key] = seen.get(key, 0) + 1
    top = sorted(seen.items(), key=lambda kv: -kv[1])[:max_pairs]
    return {'overlapping_box_pairs': n, 'top_overlapping_label_pairs': [{'a': a, 'b': b, 'count': c} for (a, b), c in top]}

# ---------- KB: COCO annotations + caption cues ----------
with open(DATA/'approved_annotations/kb_instances.json') as f: kb_inst = json.load(f)
cat_name = {c['id']: c['name'] for c in kb_inst['categories']}
by_img = defaultdict(list)
for a in kb_inst['annotations']:
    x, y, w, h = a['bbox']
    if w > 0 and h > 0 and not a.get('iscrowd', 0):
        by_img[a['image_id']].append((cat_name[a['category_id']], x, y, w, h))
count_by_img = defaultdict(lambda: defaultdict(int))
for a in kb_inst['annotations']:
    count_by_img[a['image_id']][cat_name[a['category_id']]] += 1
cap_by_img = caps.groupby('image_id')['caption_raw'].apply(list).to_dict()

records = {}
for iid in ids['knowledge_base']:
    cnt = dict(sorted(count_by_img[iid].items()))
    records[iid] = {
        'image_id': iid, 'split': 'knowledge_base',
        'objects': cnt, 'object_count': int(sum(cnt.values())),
        'objects_source': 'COCO instances_train2017 (human annotation)',
        **caption_cues(cap_by_img[iid]),
        'attributes_context_source': 'keywords in the 5 KB captions (heuristic, not visual ground truth)',
        'relations': overlap_pairs(by_img[iid]),
        'relations_source': 'COCO bbox geometry (proxy, not semantic)',
    }
print('KB records:', len(records))

KB records: 10000


## 3. Validation and test images: object detector

Then save all 10,200 records.

In [ ]:
# ---------- validation + test: detector ----------
from transformers import DetrImageProcessor, DetrForObjectDetection
DET_NAME, DET_THR = 'facebook/detr-resnet-50', 0.7
det_proc = DetrImageProcessor.from_pretrained(DET_NAME)
det = DetrForObjectDetection.from_pretrained(DET_NAME).to(device).eval()

@torch.inference_mode()
def detect(image):
    inp = det_proc(images=image, return_tensors='pt').to(device)
    out = det(**inp)
    r = det_proc.post_process_object_detection(out, threshold=DET_THR, target_sizes=[image.size[::-1]])[0]
    objs = []
    for s, l, b in zip(r['scores'].tolist(), r['labels'].tolist(), r['boxes'].tolist()):
        name = det.config.id2label[l]
        if name in ('N/A', 'N/A '): continue
        x0, y0, x1, y1 = b
        objs.append((name, x0, y0, x1 - x0, y1 - y0, s))
    return objs

t0 = time.perf_counter()
for split in ['validation', 'test']:
    for iid in ids[split]:
        objs = detect(load_image(iid))
        cnt = {}
        for o in objs: cnt[o[0]] = cnt.get(o[0], 0) + 1
        records[iid] = {
            'image_id': iid, 'split': split,
            'objects': dict(sorted(cnt.items())), 'object_count': len(objs),
            'objects_source': f'{DET_NAME}, score >= {DET_THR}',
            'colors': None, 'shapes': None, 'context': None,
            'attributes_context_source': 'not extracted (test captions are never read)',
            'relations': overlap_pairs([o[:5] for o in objs]),
            'relations_source': 'detector bbox geometry (proxy, not semantic)',
        }
print('detector time: %.1fs' % (time.perf_counter() - t0))

assert len(records) == 10200 and set(records) == set(fname)
with open(META/'image_metadata.json', 'w') as f:
    json.dump([records[i] for i in sorted(records)], f)
flat = pd.DataFrame([{
    'image_id': r['image_id'], 'split': r['split'], 'object_count': r['object_count'],
    'objects': json.dumps(r['objects']), 'colors': json.dumps(r['colors']), 'shapes': json.dumps(r['shapes']),
    'context': json.dumps(r['context']), 'overlapping_box_pairs': r['relations']['overlapping_box_pairs'],
    'objects_source': r['objects_source']} for r in records.values()]).sort_values('image_id')
flat.to_csv(META/'image_metadata.csv', index=False)

summary = {
    'records': len(records),
    'per_split': flat.groupby('split').size().to_dict(),
    'mean_objects_per_image': flat.groupby('split')['object_count'].mean().round(2).to_dict(),
    'detector': {'name': DET_NAME, 'score_threshold': DET_THR, 'transformers': transformers.__version__, 'torch': torch.__version__},
    'caveat': 'colours/shapes/context are caption-keyword heuristics; relations are bbox-overlap proxies',
}
(QA/'metadata_summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


detector time: 26.0s
{
  "records": 10200,
  "per_split": {
    "knowledge_base": 10000,
    "test": 100,
    "validation": 100
  },
  "mean_objects_per_image": {
    "knowledge_base": 7.21,
    "test": 6.71,
    "validation": 9.27
  },
  "detector": {
    "name": "facebook/detr-resnet-50",
    "score_threshold": 0.7,
    "transformers": "5.22.1",
    "torch": "2.14.1+cu130"
  },
  "caveat": "colours/shapes/context are caption-keyword heuristics; relations are bbox-overlap proxies"
}
